# Model Training

Training-only notebook for the final artificial neural network used to predict
vitrimer self-healing efficiency.

This version contains no plotting, exploratory-data-analysis, prediction-profile,
or visualization code.

The preprocessing and training configuration is aligned with the current
manuscript:

- Tg is retained as an input feature.
- Healing temperature is retained as an input feature.
- Only catalyst content and healing time receive targeted feature engineering.
- All data-dependent preprocessing objects are fitted using training data only.
- Final input dimensionality after PCA: 15.
- Random seed: 123.
- Activation: Leaky ReLU.
- Optimizer: RMSprop.
- Learning rate: 0.001.
- Batch size: 24.
- Maximum epochs: 9000.
- Early-stopping patience: 300.


## 1. Imports and reproducibility


In [ ]:
import os
import random
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input, LeakyReLU
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import RMSprop

from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.decomposition import PCA

# ============================================================
# Reproducibility
# ============================================================

SEED = 123

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


## 2. File paths and final model settings


In [ ]:
# ============================================================
# File paths
# ============================================================

DATA_PATH = Path("Raw Dataset_dir")

OUTPUT_DIR = Path("model")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = OUTPUT_DIR / "Final_LeakyReLU_RMSprop_Seed123.keras"
PREPROCESSOR_PATH = OUTPUT_DIR / "Final_Preprocessing_Seed123.joblib"

# ============================================================
# Dataset settings
# ============================================================

TARGET_COLUMN = "Self-healing Effficiency (%)"

# The manuscript uses 41 completely unseen samples as the
# independent test set.
N_TEST_SAMPLES = 41

# 20% of the remaining model-development data is reserved
# internally for validation.
VALIDATION_SIZE = 0.20

# ============================================================
# Preprocessing settings
# ============================================================

CORRELATION_THRESHOLD = 0.80
N_PCA_COMPONENTS = 15

# ============================================================
# Final ANN hyperparameters
# ============================================================

LEARNING_RATE = 0.001
BATCH_SIZE = 24
MAX_EPOCHS = 9000
PATIENCE = 300


## 3. Load the dataset


In [ ]:
data = pd.read_excel(DATA_PATH)
data.columns = data.columns.astype(str).str.strip()

if TARGET_COLUMN not in data.columns:
    raise KeyError(
        f"Target column '{TARGET_COLUMN}' was not found in the input file."
    )


## 4. Remove obsolete engineered features

Older versions of the workflow contained engineered Tg and temperature features.
They are not used in the current model. The original Tg and healing-temperature
features are retained.


In [ ]:
# Remove engineered variables from older versions of the model, if present.
# IMPORTANT:
#   - "Tg (C)" is NOT removed.
#   - "Temperature (C)" is NOT removed.

OBSOLETE_ENGINEERED_COLUMNS = [
    "Alternate Tg (C)",
    "Alternate Temperature",
    "Temperature Average",
    "Alternate Temperature Average",
]

obsolete_present = [
    col for col in OBSOLETE_ENGINEERED_COLUMNS
    if col in data.columns
]

data = data.drop(columns=obsolete_present)


## 5. Independent test split and validation split

The split is performed before fitting the imputer, scaler, correlation filter,
PowerTransformer, or final PCA.

If a `Split` column is included in the released dataset, its labels are used to
reproduce the predetermined independent test set. Otherwise, the code falls back
to a deterministic 41-sample split using seed 123.


In [ ]:
# ============================================================
# Separate X and y
# ============================================================

y_all = data[TARGET_COLUMN].astype(float).copy()
X_all = data.drop(columns=[TARGET_COLUMN]).copy()

# ============================================================
# Independent test split
# ============================================================

if "Split" in X_all.columns:
    split_labels = X_all["Split"].astype(str).str.strip().str.lower()
    X_all = X_all.drop(columns=["Split"])

    test_mask = split_labels.eq("test")
    development_mask = ~test_mask

    X_development_raw = X_all.loc[development_mask].copy()
    y_development = y_all.loc[development_mask].copy()

    X_test_raw = X_all.loc[test_mask].copy()
    y_test = y_all.loc[test_mask].copy()

else:
    X_development_raw, X_test_raw, y_development, y_test = train_test_split(
        X_all,
        y_all,
        test_size=N_TEST_SAMPLES,
        random_state=SEED,
    )

# ============================================================
# Validation split
# ============================================================

X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_development_raw,
    y_development,
    test_size=VALIDATION_SIZE,
    random_state=SEED,
)


## 6. Targeted feature engineering

Only the two transformations retained in the current manuscript are applied:

\[
C_{cata}^{*}=-C_{cata}
\]

\[
t_{heal}^{*}=-\sqrt{t_{heal}}
\]

The original catalyst-content and healing-time columns are replaced by these
engineered variables. Tg and healing temperature are not transformed.


In [ ]:
def engineer_features(df):
    df = df.copy()

    catalyst_candidates = [
        "Catalyst (%wt)",
        "Catalyst wt.%",
        "Catalyst (wt%)",
    ]

    time_candidates = [
        "Required Time (min)",
        "Healing Time (min)",
        "Time (min)",
    ]

    catalyst_col = next(
        (col for col in catalyst_candidates if col in df.columns),
        None
    )

    time_col = next(
        (col for col in time_candidates if col in df.columns),
        None
    )

    if catalyst_col is None:
        raise KeyError(
            "Catalyst-content column was not found. "
            f"Expected one of: {catalyst_candidates}"
        )

    if time_col is None:
        raise KeyError(
            "Healing-time column was not found. "
            f"Expected one of: {time_candidates}"
        )

    catalyst = pd.to_numeric(df[catalyst_col], errors="coerce")
    healing_time = pd.to_numeric(df[time_col], errors="coerce")

    if (healing_time.dropna() < 0).any():
        raise ValueError("Healing time cannot contain negative values.")

    # Eq. (3)
    df["Transformed Catalyst (%wt)"] = -catalyst

    # Eq. (4)
    df["Transformed Healing Time"] = -np.sqrt(healing_time)

    # The transformed variables replace their original versions.
    df = df.drop(columns=[catalyst_col, time_col])

    # Remove equivalent engineered columns from older preprocessing versions.
    old_equivalent_columns = [
        "Alternate Catalyst (%wt)",
        "Time",
    ]

    old_equivalent_columns = [
        col for col in old_equivalent_columns
        if col in df.columns
    ]

    df = df.drop(columns=old_equivalent_columns)

    return df


X_train_eng = engineer_features(X_train_raw)
X_val_eng = engineer_features(X_val_raw)
X_test_eng = engineer_features(X_test_raw)


## 7. KNN imputation

The KNN imputer is fitted only on the training subset and is subsequently applied
without refitting to validation and independent-test samples.


In [ ]:
imputer = KNNImputer(n_neighbors=2)

X_train_imp = pd.DataFrame(
    imputer.fit_transform(X_train_eng),
    columns=X_train_eng.columns,
    index=X_train_eng.index,
)

X_val_imp = pd.DataFrame(
    imputer.transform(X_val_eng),
    columns=X_train_eng.columns,
    index=X_val_eng.index,
)

X_test_imp = pd.DataFrame(
    imputer.transform(X_test_eng),
    columns=X_train_eng.columns,
    index=X_test_eng.index,
)


## 8. Standardization

StandardScaler is fitted using the training subset only.


In [ ]:
scaler = StandardScaler()

X_train_std = pd.DataFrame(
    scaler.fit_transform(X_train_imp),
    columns=X_train_imp.columns,
    index=X_train_imp.index,
)

X_val_std = pd.DataFrame(
    scaler.transform(X_val_imp),
    columns=X_train_imp.columns,
    index=X_val_imp.index,
)

X_test_std = pd.DataFrame(
    scaler.transform(X_test_imp),
    columns=X_train_imp.columns,
    index=X_test_imp.index,
)


## 9. Correlation filtering

Absolute Pearson correlations are calculated from the standardized training data
only. For each pair with \(|PCC| > 0.8\), the later feature in the matrix is
removed. The same retained feature set is then used for validation and test data.


In [ ]:
corr_matrix = X_train_std.corr().abs()

upper_triangle = corr_matrix.where(
    np.triu(
        np.ones(corr_matrix.shape, dtype=bool),
        k=1
    )
)

correlated_columns_to_drop = [
    column
    for column in upper_triangle.columns
    if any(upper_triangle[column] > CORRELATION_THRESHOLD)
]

X_train_filtered = X_train_std.drop(
    columns=correlated_columns_to_drop
)

X_val_filtered = X_val_std.drop(
    columns=correlated_columns_to_drop
)

X_test_filtered = X_test_std.drop(
    columns=correlated_columns_to_drop
)


## 10. Power transformation

The Yeo–Johnson PowerTransformer is fitted only on the retained training features.


In [ ]:
power_transformer = PowerTransformer(method="yeo-johnson")

X_train_pt = pd.DataFrame(
    power_transformer.fit_transform(X_train_filtered),
    columns=X_train_filtered.columns,
    index=X_train_filtered.index,
)

X_val_pt = pd.DataFrame(
    power_transformer.transform(X_val_filtered),
    columns=X_train_filtered.columns,
    index=X_val_filtered.index,
)

X_test_pt = pd.DataFrame(
    power_transformer.transform(X_test_filtered),
    columns=X_train_filtered.columns,
    index=X_test_filtered.index,
)


## 11. Final PCA

The second PCA is fitted exclusively on the processed training features and
reduces the final model input to 15 principal components.


In [ ]:
if X_train_pt.shape[1] < N_PCA_COMPONENTS:
    raise ValueError(
        f"Only {X_train_pt.shape[1]} features remain after preprocessing, "
        f"which is fewer than the requested {N_PCA_COMPONENTS} PCA components."
    )

pca = PCA(
    n_components=N_PCA_COMPONENTS,
    random_state=SEED
)

X_train = pca.fit_transform(X_train_pt)
X_val = pca.transform(X_val_pt)
X_test = pca.transform(X_test_pt)


## 12. Final ANN

Architecture:

15 → 256 → 128 → 64 → 32 → 16 → 8 → 4 → 2 → 1

All eight hidden layers use Leaky ReLU. The output layer is linear.


In [ ]:
tf.keras.backend.clear_session()

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

model = Sequential([
    Input(shape=(N_PCA_COMPONENTS,)),

    Dense(256),
    LeakyReLU(),

    Dense(128),
    LeakyReLU(),

    Dense(64),
    LeakyReLU(),

    Dense(32),
    LeakyReLU(),

    Dense(16),
    LeakyReLU(),

    Dense(8),
    LeakyReLU(),

    Dense(4),
    LeakyReLU(),

    Dense(2),
    LeakyReLU(),

    Dense(1, activation="linear"),
])

model.compile(
    optimizer=RMSprop(learning_rate=LEARNING_RATE),
    loss="mae",
    metrics=["mae"],
)


## 13. Model training


In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=PATIENCE,
    mode="min",
    restore_best_weights=True,
)

history = model.fit(
    X_train,
    y_train.to_numpy(),
    validation_data=(X_val, y_val.to_numpy()),
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping],
    verbose=1,
)


## 14. Save the final model and preprocessing objects


In [ ]:
model.save(MODEL_PATH)

preprocessing_pipeline = {
    "seed": SEED,
    "target_column": TARGET_COLUMN,
    "raw_input_columns": list(X_train_raw.columns),
    "engineered_input_columns": list(X_train_eng.columns),
    "retained_columns_after_correlation_filter": list(X_train_filtered.columns),
    "dropped_correlated_columns": correlated_columns_to_drop,
    "correlation_threshold": CORRELATION_THRESHOLD,
    "imputer": imputer,
    "scaler": scaler,
    "power_transformer": power_transformer,
    "pca": pca,
}

joblib.dump(
    preprocessing_pipeline,
    PREPROCESSOR_PATH
)
